> **中文原创篇** —— 本篇非上游译文，为中文社区自建教学内容；API 行为以仓库实际代码为准（`semantica.reasoning`、`rdflib`、`owlrl`）。

# 张三该涨工资吗？——一个例子看懂 OWL DL、OWL RL、Datalog 与统一推理门面

## 概览

公司规定「经理 = 公司员工，且负责至少一个项目」，老板对张三说「你之后负责 semantica-cn 项目」。**从头到尾没人写下「张三是经理」**——本篇让机器自己把它推出来，顺手算出张三该不该涨工资。

同一个制度，四种写法各写一遍，正好对上落地路线：

| 语言/工具 | 制度写成什么 | 本篇怎么跑 |
|---|---|---|
| **OWL DL** | 双向定义（「等于」） | Part 2：rdflib 只解析不推理（仓库无 DL 推理机） |
| **OWL RL** | 单向分类（「都算」） | Part 3：`owlrl` 纯类库真实物化 |
| **Datalog** | Horn 子句（`head :- body`） | Part 4：`DatalogReasoner` 高效求值（不记账） |
| **统一推理门面（Reasoner）** | 产生式规则（`IF..THEN`） | Part 5：一次性推理，结论自带依据 + 解释；Part 6 登记决策 |

### 🎯 学习目标

- 写出「经理」的 OWL DL 定义与 OWL RL 合法改写，说清两者差在哪
- 用 `owlrl` 类库对 OWL 2 RL 本体做真实物化，验证张三的经理身份与工资被自动推出
- 分别用 `DatalogReasoner`（Horn 子句）与 `Reasoner` 门面（产生式规则）推出张三的应得工资，并比较产物差异
- 用 `ExplanationGenerator` 生成结论解释，用 `ContextGraph.record_decision()` 登记决策并存档溯源

### 📚 前置

- [14_Datalog_Style_Reasoning.ipynb](./14_Datalog_Style_Reasoning.ipynb) —— `DatalogReasoner` 基础（EDB/IDB/定点求值）
- [14_Ontology.ipynb](../introduction/14_Ontology.ipynb) —— 本体是什么
- [26_Semantic_Layer_Basics.ipynb](../introduction/26_Semantic_Layer_Basics.ipynb) —— 图、本体、映射如何拼装

### 先看主线

**制度与事实分开存放，是语义技术的第一个动作。** 制度（员工手册）进规则库，事实（系统数据）进数据库，两边变动互不牵连：调薪只改制度，换岗只加事实。张三的疑问就是一场推理，七个 Part 一段对账：

**案情（Part 1）**

1. 制度三条、事实三条，张三问：我该涨工资吗？

**本体两式（Part 2–3）**

2. OWL DL：制度写成充要定义；rdflib 只解析不推理
3. OWL RL：owlrl 纯类库真实物化——张三的经理身份和工资全推出来

**规则两式（Part 4–5）**

4. Datalog：Horn 子句高效求值（不记账）
5. 统一推理门面：产生式规则一次性推理，结论自带依据，解释直接可用

**决策（Part 6）**

6. 推理给归因 → 人拍板 → 登记存档（record_decision + 溯源）

**对账（Part 7）**

7. 四种写法各自的答案与边界

依赖：`semantica-cn`（分发包名，`Reasoner`/`DatalogReasoner` 在核心能力里）、`rdflib`（核心依赖，解析 Turtle 与 SPARQL 查询）和 `owlrl`（OWL 2 RL 物化类库，Part 3 用，纯 Python 无需任何服务）。执行下面这格直接从 fork 仓库安装，与 Jupyter 启动位置无关；已经装过的环境直接跳过。

In [11]:
# 安装依赖：semantica-cn（fork 仓库）+ owlrl（OWL 2 RL 物化，Part 3 用）；已装可跳过
!pip install -qU "semantica-cn @ git+https://github.com/OntologyAgent/semantica-cn.git"
!pip install -q owlrl

from importlib.metadata import version
import rdflib
print("semantica-cn 就绪：", version("semantica-cn"), "| rdflib", rdflib.__version__)

semantica-cn 就绪： 0.8.0 | rdflib 7.6.0


In [12]:
# ── 案情数据：制度与事实分开摆 ──────────────────────────────────────────────
policy = {                    # 制度（员工手册）
    "manager_def": "经理 = 公司员工，且负责至少一个项目",   # 规定①
    "manager_pay": 10000,       # 规定②：经理岗位工资
    "staff_pay":   6000,        # 规定③：普通员工岗位工资
}

facts = [                     # 事实（系统里的数据，Part 4/5 原样喂给推理引擎）
    "Employee(张三)",               # 张三是（普通）员工
    "Project(semantica-cn)",        # semantica-cn 是项目
    "Leads(张三, semantica-cn)",     # 老板今天说的 ← 唯一的新事实
]

print("制度：", policy["manager_def"])
print("事实：", *facts, sep="\n      ")

制度： 经理 = 公司员工，且负责至少一个项目
事实：
      Employee(张三)
      Project(semantica-cn)
      Leads(张三, semantica-cn)


## Part 1 —— 案情：制度、事实、一个问题

**制度（员工手册 → 规则库）**

| 编号 | 规定 |
|---|---|
| ① | 经理 = 公司员工，且负责至少一个项目 |
| ② | 经理岗位工资 10000 元/月 |
| ③ | 普通员工岗位工资 6000 元/月 |

**事实（系统数据 → 数据库）**

- 张三：入职 3 个月，岗位普通员工，现薪 6000
- 今天老板说：「张三之后负责 semantica-cn 项目」
- semantica-cn 是项目

**难点**：没有一个人写下「张三是经理」。老板说的是「你负责项目」，制度里写的是「经理 = 员工且负责项目」。要把「负责」翻成「该涨工资」，机器得完成两次跳转：**负责过项目 → 是经理 → 应得 10000**。这两跳就是推理。

## Part 2 —— OWL DL：制度写成充要定义

OWL 里规定①写成「Manager **等于**『员工 且 负责至少一个项目』」——双向充要。规定②用 `owl:hasValue` 限制挂上（所有经理月薪 10000）。

**Semantica 的现状先交代清楚**：仓库没有 DL 推理机（HermiT/Pellet 都没集成）。下面这格只做**解析**（rdflib），不做推理。DL 的强项在制度设计期——查制度自相矛盾、自动算岗位分类树——那要外接工具，见本 Part 末尾。

In [13]:
from rdflib import Graph

MANAGER_DL = """
@prefix :     <https://example.org/company/> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Manager a owl:Class ;                # 经理
    owl:equivalentClass [             # 「等于」：双向充要定义
        a owl:Class ;
        owl:intersectionOf (          # 「并且」
            :Employee                 # 条件 A：公司员工
            [ a owl:Restriction ;     # 条件 B：一条限制
              owl:onProperty :leads ;
              owl:someValuesFrom :Project ]  # 至少负责一个项目
        ) ;
    ] ;
    rdfs:subClassOf [                 # 规定②：经理 ⇒ 月薪 10000
        a owl:Restriction ;
        owl:onProperty :salary ;
        owl:hasValue 10000 ] .
"""

g = Graph()
g.parse(data=MANAGER_DL, format="turtle")
print(f"OWL DL 定义解析成功：{len(g)} 条三元组")
print(g.serialize(format="turtle"))

OWL DL 定义解析成功：15 条三元组
@prefix : <https://example.org/company/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:Manager a owl:Class ;
    rdfs:subClassOf [ a owl:Restriction ;
            owl:hasValue 10000 ;
            owl:onProperty :salary ] ;
    owl:equivalentClass [ a owl:Class ;
            owl:intersectionOf ( :Employee [ a owl:Restriction ;
                        owl:onProperty :leads ;
                        owl:someValuesFrom :Project ] ) ] .




这份定义交给 DL 推理机能做两件 Datalog 做不到的事：

- **反向补全**：从「X 是经理」推出「X 负责某个项目」——哪怕数据里没写他负责谁，证明器在内部造一个看不见的项目（存在见证）
- **一致性检查**：若再规定「实习生 disjointWith 经理」而某人两者都是，设计期直接报制度自相矛盾

外部工具的做法（需要 Java，只给形状，本篇不执行）：

```python
# from owlready2 import get_ontology, sync_reasoner_pellet
# onto = get_ontology("manager_dl.ttl").load()
# with onto:
#     sync_reasoner_pellet()   # Pellet/HermiT 跑 tableau
```

Semantica 的取舍：DL 留给设计期外援，运行期推理走自家规则引擎——名实相符。

## Part 3 —— OWL RL：用 owlrl 直接物化

**RL 不是新语言**——还是同一份 Turtle，加了一份禁写清单：`owl:equivalentClass` 右边不许放匿名描述（要凭空造「看不见的项目」，而 RL 的规则只会搬已有数据）。合法改写：识别方向的匿名交集放 `rdfs:subClassOf` **左边**；规定②的 `hasValue` 限制放超类侧，RL 也合法。

**`owlrl` 是纯 Python 类库**，`pip install owlrl` 即有，不装任何数据库、不起任何服务：它把 OWL 2 RL 规范内置的几十条规则在 rdflib 图上跑到不动点，推导出的三元组**物理写进图里**（物化）。生产级的大图物化才需要换 GraphDB / RDFox 这类服务器（Part 7 对比）。

In [14]:
MANAGER_RL = """
@prefix :     <https://example.org/company/> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

# 识别方向（RL 合法）：匿名交集放 subClassOf 左边
# 「是员工 且 负责至少一个项目」的东西，都算经理
[ owl:intersectionOf (
      :Employee
      [ a owl:Restriction ;
        owl:onProperty :leads ;
        owl:someValuesFrom :Project ] ) ;
  rdfs:subClassOf :Manager ] .

# 蕴含方向的合法半句（另一半 Manager ⊑ ∃leads.Project RL 语法禁止）
:Manager rdfs:subClassOf :Employee .

# 负责的主体是员工（domain 公理，RL 合法）
:leads rdfs:domain :Employee .

# 规定②的 RL 合法写法：经理 ⇒ 月薪 10000（hasValue 限制，超类侧允许）
:Manager rdfs:subClassOf [
    a owl:Restriction ; owl:onProperty :salary ; owl:hasValue 10000 ] .
"""

FACTS = """
@prefix :     <https://example.org/company/> .

:semantica-cn a :Project .
:张三 :leads :semantica-cn .   # 老板那句话；注意：没断言张三是员工，更没说是经理
"""

g_rl = Graph()
g_rl.parse(data=MANAGER_RL, format="turtle")
g_rl.parse(data=FACTS, format="turtle")
print(f"本体 + 事实共 {len(g_rl)} 条三元组")
before = list(g_rl.query("SELECT ?c WHERE { :张三 a ?c }"))
print("物化前张三的类型：", before or "空——一条类型都没写")

本体 + 事实共 17 条三元组
物化前张三的类型： 空——一条类型都没写


In [15]:
import owlrl

owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(g_rl)   # 规则跑到不动点，推导三元组物理写入
print(f"物化后：{len(g_rl)} 条三元组")

print("张三的类型：", [str(r.c).split("/")[-1] for r in g_rl.query(
    "SELECT ?c WHERE { :张三 a ?c . FILTER(isIRI(?c)) }")])
print("应得工资：", [r.s.toPython() for r in g_rl.query("SELECT ?s WHERE { :张三 :salary ?s }")])

物化后：156 条三元组
张三的类型： ['Employee', 'Manager']
应得工资： [10000]


对照着看，三条制度全部自动兑现，而数据里只写了一句「张三负责 semantica-cn」：

- **type(张三, Employee)**：没人断言——`leads` 的 domain 公理（`:leads rdfs:domain :Employee`）推出来的
- **type(张三, Manager)**：识别公理的功劳。RL 引擎内部是三条规范规则接力：cls-svf1（张三满足「负责 ≥1 项目」的匿名限制）→ cls-int1（再加上 Employee，两个条件齐了）→ cax-sco（匿名类 ⊑ Manager 传播）
- **salary 10000**：hasValue 公理（Manager ⊑ 月薪 10000）由 cls-hv1 规则兑现——规定②直接写进了本体

三点工程提醒：

- **17 条 → 156 条**：多出的 139 条里大量是技术性三元组（owl:Thing、限制节点的类型、rdfs:member 之类）——这是**物化的空间代价**；数据一变还要维护这些推导，删尤其难（真值维护问题）
- 纯 Python 的 owlrl 在大图上会慢——那是换 **GraphDB**（Ontotext 的免费版服务器，Docker 镜像或独立安装包，内置 owl-rl ruleset）或 **RDFox**（商业授权，Datalog 物化性能天花板）的时机。注意两者都是**需要部署的服务器**，不是类库
- owlrl 的规则表是 W3C 规范定死的——同一份 RL 本体在 owlrl、GraphDB、RDFox 里得到一致的推理结果，这就是**标准**的价值

## Part 4 —— Datalog：Horn 子句，高效求值

经典 Datalog 语法一行学会：**`结论 :- 前提1, 前提2.`** 读作「如果右边全成立，左边就成立」；大写开头是变量，逗号是「并且」。这个形态叫 **Horn 子句（Horn clause）**，逻辑编程（Prolog/Datalog）的标准记法。

`DatalogReasoner` 用半朴素（semi-naive）定点求值——只检查「上轮新增的事实」能触发什么，大图上比朴素匹配快得多，还支持递归规则。**代价：它不记账**——推导出的结论不带「哪条规则、哪些前提」。要可解释的推理？这正是 Part 5 存在的理由。

In [16]:
from semantica.reasoning import DatalogReasoner

datalog = DatalogReasoner()

# ── 事实（GraphBuilder 的 dict 格式直接喂）──
datalog.add_fact({"id": "张三", "type": "Employee"})
datalog.add_fact({"id": "semantica-cn", "type": "Project"})
datalog.add_fact({"source": "张三", "target": "semantica-cn", "type": "leads"})

# ── 制度（规定①②翻译成 Horn 子句）──
datalog.add_rule("manager(X) :- employee(X), leads(X, Y), project(Y).")  # 规定①
datalog.add_rule("salary(X, 10000) :- manager(X).")                      # 规定②

datalog.derive_all()   # 半朴素求值，跑到不动点

print("经理认定：", datalog.query("manager(?X)"))
print("应得工资：", datalog.query("salary(?X, ?S)"))

🔄 Semantica is reasoning: Starting semi-naive fixpoint evaluation 🤔 reasoning DatalogReasoner |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

经理认定： [{'X': '张三'}]
应得工资： [{'X': '张三', 'S': '10000'}]


结论对了，但注意产物形态：`query()` 只回**变量绑定**——没有「这条结论用了哪条规则、哪些前提」。求值引擎给你答案，不给你证明。

同一个制度，Part 5 换一套记法再推一遍：逻辑内容一字不变，产物却完全不同。

## Part 5 —— 统一推理门面（Reasoner）：一次性推理，结论自带依据

规则形态换成**产生式规则（production rule）**：**`IF 前提1 AND 前提2 THEN 结论`**——前件/LHS（IF 部分）+ 后件/RHS（THEN 部分），`?x` 是变量。

`Reasoner` 是**统一推理门面**（Facade 门面模式：统一入口封装多种推理策略）——`forward_chain()` 一次性跑到不动点，**每条结论自带「用了哪条规则、哪些前提」**，推理和记录是同一个动作。对比 Part 4：同一个制度，两套记法（Horn 子句 ↔ 产生式规则），逻辑等价可互译；区别在产物——门面记账，Datalog 只给答案。

In [17]:
from semantica.reasoning import Reasoner

engine = Reasoner()

# ── 事实（案情里的三条）──
engine.add_fact("Employee(张三)")
engine.add_fact("Project(semantica-cn)")
engine.add_fact("Leads(张三, semantica-cn)")

# ── 制度（规定①②逐条翻译成产生式规则）──
engine.add_rule("IF Employee(?x) AND Leads(?x, ?y) AND Project(?y) THEN Manager(?x)")  # 规定①
engine.add_rule("IF Manager(?x) THEN Salary(?x, 10000)")                               # 规定②

results = engine.forward_chain()   # 一次性推理：跑到不动点，结论自带依据

print("── 全部结论 ──")
for res in results:
    print(f"  {res.conclusion}")
    print(f"    规则: {res.rule_used.name} | 前提: {res.premises}")

🔄 Semantica is reasoning: Performing forward chaining 🤔 reasoning Reasoner |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

── 全部结论 ──
  Manager(张三)
    规则: Rule 1 | 前提: ['Employee(张三)', 'Leads(张三, semantica-cn)', 'Project(semantica-cn)']
  Salary(张三, 10000)
    规则: Rule 2 | 前提: ['Manager(张三)']


`forward_chain()` 的返回值就是**决策过程的账本**：

- `Manager(张三)`，前提 = 三个原始事实 ← 规定①
- `Salary(张三, 10000)`，前提 = `Manager(张三)` ← 规定②（同一轮里链式接力）

**「张三是经理」「应得 10000」都没写在任何表里——但每条结论从诞生的那一刻就带着推导依据。** 对照张三现薪 6000：该涨。

工程红利：下月调薪只改规则一行，不动数据；张三换岗只加一条事实，不动制度。

### 针对结论解释：推理结果直接可用

`ExplanationGenerator` 直接吃 `forward_chain()` 返回的 `InferenceResult`——结论已经带着规则和前提，**不需要任何手工组装**。挑出工资那条结论：

In [18]:
from semantica.reasoning import ExplanationGenerator

salary_res = next(res for res in results if res.conclusion.startswith("Salary"))
gen = ExplanationGenerator(detail_level="detailed")

explanation = gen.generate_explanation(salary_res)
print("自然语言:", explanation.natural_language)
print("推理路径步骤数:", len(explanation.reasoning_path.steps) if explanation.reasoning_path else 0)

🔄 Semantica is reasoning: Generating explanation for reasoning result 🤔 reasoning ExplanationGenerator |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

自然语言: Given the premises: Manager(张三), we conclude: Salary(张三, 10000) using rule 'Rule 2'.
推理路径步骤数: 2


到这里，「推理 → 解释」是一个连续动作：`forward_chain()` 一次出全部结论，挑任何一条都能立刻答「为什么」。三条路线的解释能力在此分化：

- **规则推理（本格）**：结论自带 `rule_used` + `premises`——推理时即记录
- **owlrl 物化**：查不了——156 条推导与原始无差别（物化的固有代价）
- **生产服务器**：RDFox / GraphDB 内建推导树与证明插件；DL 推理机给最小 justification
- 实测小坑：`backward_chain("Salary(张三, 10000)")` 对这条两跳链给出的证明不完整（前提里包含结论自身）——多跳结论的解释，用 forward 结果更可靠

再往后一步是业务动作：HR 据此调薪。这类「我们据此做了什么」的记录用 `ContextGraph.record_decision()` 把决策本身存成一等对象（带溯源）——规则解释答「事实怎么来的」，决策登记答「我们据此做了什么」。

## Part 6 —— 决策与登记：推理给归因，人拍板，登记存档

推理层的产出到此为止：**归因**——带依据链的结论（Part 5 的 `InferenceResult`）。但「该不该涨」是业务决策，由人（或业务系统）拍板；拍板之后，把「决策 + 归因链」一起登记在案，将来任何时刻可查、可审计：

1. **推理归因**：`Manager(张三)` ← 员工 + 负责 semantica-cn + 是项目（规定①）；`Salary 10000` ← 规定②
2. **人拍板**：HR 依据归因决定调薪
3. **登记存档**：`record_decision()` 把决策记成图上的一等节点，`track_entity()` 挂上溯源链

In [19]:
from semantica.context import ContextGraph
from semantica.provenance import ProvenanceManager

cg = ContextGraph()
pm = ProvenanceManager()

# ① 登记决策：拍板了什么、依据是什么、结果是什么
decision_id = cg.record_decision(
    category="salary_adjustment",
    scenario="张三负责 semantica-cn 项目，触发经理岗调薪",
    reasoning="规定①(经理=员工且负责≥1项目) + 规定②(经理月薪10000)；"
              "推理推出 manager(张三) → salary(张三, 10000)",
    outcome="张三月薪 6000 → 10000",
    confidence=1.0,
    entities=["张三", "semantica-cn"],
    decision_maker="HR-系统",
)

# ② 给决策挂溯源档案（来源 + 防篡改校验链）
pm.track_entity(decision_id, source="salary-policy-inference",
                metadata={"derived_from": ["manager(张三)", "salary 规定②"]})

summary = cg.get_decision_summary()
print("决策档案号：", decision_id)
print("登记决策数：", summary["total_decisions"])
print("溯源来源：", pm.get_lineage(decision_id)["source_documents"])

决策档案号： 43cae113-56f5-402c-bedd-d38feef0cd27
登记决策数： 1
溯源来源： ['salary-policy-inference']


半年后有人问「张三凭什么拿 10000」，两层账本分别作答：

- **推理的账**（Part 5）：`ExplanationGenerator` 解释「为什么推出 manager(张三)」
- **管理的账**（本格）：`cg.get_decision_summary()` 看全部决策分布；`pm.get_lineage(decision_id)` 出这条决策的来源、时间与防篡改校验链；`pm.export(...)` 还能导出 W3C PROV-O 标准文件

分工一句话：规则解释答「事实怎么来的」，决策登记答「我们据此做了什么」。后者带 checksum 防篡改链，审计场景的硬需求。

## Part 7 —— 对账：四种写法各自的答案

| | OWL DL | OWL RL（owlrl） | Datalog | Reasoner 门面 |
|---|---|---|---|---|
| 制度写成 | 双向定义（「等于」） | 单向分类 + 公理（「都算」） | Horn 子句（`head :- body`） | 产生式规则（`IF..THEN`） |
| 张三的问题 | 制度没错就能答，但 Semantica 不带推理机 | ✅ 物化后查出该涨 | ✅ 直接算出该涨 | ✅ 直接算出该涨 |
| 结论带依据 | —（需外接推理机） | ❌ 物化无差别，查不了 | ❌ 不记账 | ✅ `rule_used` + `premises` |
| 决策记录 | — | — | — | ✅ 配 `ExplanationGenerator` + `record_decision` |
| 形态 | 外部工具（需 Java） | 纯类库，pip 即用 | Semantica 原生引擎 | Semantica 原生引擎 |

**为何 DatalogReasoner 不在 Reasoner 门面下**：门面吃产生式规则、产 `InferenceResult`（结论自带前提）；DatalogReasoner 吃 Horn 子句、半朴素求值产裸事实（不记账）——语法、产物契约、求值模型三者都不同，故为平行工具而非门面内的策略。

**三句话带走**

1. 制度与事实分开存放——调薪改规则，换岗加事实，互不牵连
2. 「张三是经理」不在任何一张表里——它是机器从制度 + 事实里推出来的；规则推理的结论还自带依据链
3. 落地路线：推理归因用 Reasoner 门面（或 owlrl 物化），人来拍板，`record_decision()` 登记存档；DL 留给设计期外援，生产大图换 GraphDB/RDFox

### 下一步去哪

- [14_Datalog_Style_Reasoning.ipynb](./14_Datalog_Style_Reasoning.ipynb) —— Datalog 深入：递归、`ContextGraph` 装载、解释生成
- [08_Reasoning_and_Inference.ipynb](./08_Reasoning_and_Inference.ipynb) —— Semantica 推理全家桶总览
- [13_Manual_Ontology_Snowflake_Mapping.ipynb](./13_Manual_Ontology_Snowflake_Mapping.ipynb) —— 生产级语义层：治理本体 + SHACL 校验